# RDKV LongBench runs on Google Colab (A100, bf16): Team 6
These are the runs for the main results table: the code from branch `manjot/rdkv`, pinned `transformers==5.2.0`,
bf16 on one GPU (the baseline's setup), 800 records, no `--max-prompt-tokens`.

**Before running:**
1. **Runtime → Change runtime type → A100 GPU.**
2. 🔑 Secrets (left bar): your Hugging Face token as `HF_TOKEN`, with **Notebook access** on.

Results go straight to Google Drive (`MyDrive/rdkv_results_final`), so a disconnect loses nothing: run all cells again and it continues where it stopped.
Keep this tab open and the laptop awake (Terminal: `caffeinate -dims`).

In [ ]:
# 0. CONFIG
RATIOS = ["0.9", "0.75"]          # add "0.8", "0.95" if those must be rerun too
BRANCH = "manjot/rdkv"
OUT = "/content/drive/MyDrive/rdkv_results_final"   # fresh folder: only results from this exact setup

In [ ]:
# 1. GPU (expect A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Drive, token, clone the PR branch
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
token = None
for name in ["HF_TOKEN", "huggingface", "hugging_face", "hf_token", "HUGGINGFACE_TOKEN"]:
    try:
        token = userdata.get(name)
        if token:
            print("using secret:", name); break
    except Exception:
        pass
assert token, "add your Hugging Face token as a secret named HF_TOKEN (key icon, Notebook access on)"
os.environ["HF_TOKEN"] = token

REPO = "/content/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/samruddhichitnis02/298a-kv-cache-compression.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
os.makedirs(OUT, exist_ok=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Install (pinned to match the team setup)
!pip install -q "transformers==5.2.0" "kvpress==0.5.5" rouge rapidfuzz datasets peft pytest requests
import sys
if subprocess.run([sys.executable, "-c", "import fire"], capture_output=True).returncode != 0:
    !pip install -q -U "fire>=0.7"
!python -c "import torch, transformers, kvpress; print('torch', torch.__version__, '| transformers', transformers.__version__, '| kvpress OK |', torch.cuda.get_device_name(0))"

In [ ]:
# 4. Unit tests (expect: 9 passed)
!python -m pytest -q members/manjot/test_rdkv.py

In [ ]:
# 5. Build the 800-record dev set and check it matches the baseline
import json
DATA = "data/processed_longbench/processed_longbench_dev.jsonl"
if not os.path.exists(DATA) or sum(1 for _ in open(DATA)) < 800:
    if os.path.exists(DATA): os.remove(DATA)
    !python members/samruddhi/build_dev_data.py --token $HF_TOKEN
built = {json.loads(l)["record_id"]: json.loads(l)["prompt_tokens_after"] for l in open(DATA)}
base = {json.loads(l)["record_id"]: json.loads(l)["prompt_tokens"] for l in open("results/longbench/Llama-3.1-8B-Instruct_none_0.0.jsonl")}
print(f"{len(built)} records; identical to the baseline for {sum(built.get(k) == v for k, v in base.items())}/{len(base)}")

In [ ]:
# 6. FULL RUNS (bf16, one GPU). Resumes automatically after a disconnect.
for r in RATIOS:
    !python harness/run_longbench.py --press rdkv --compression-ratio {r} --output-dir {OUT}
    !python members/manjot/compare.py {OUT}/Llama-3.1-8B-Instruct_rdkv_{r}.jsonl